In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ==================== Load data ====================
X_train = np.load('Data/processed/X_train.npy')
X_test = np.load('Data/processed/X_test.npy')
y_train = np.load('Data/processed/y_train.npy')
y_test = np.load('Data/processed/y_test.npy')
le = joblib.load('Models/disease_encoder.pkl')

print(f"Original train set: {X_train.shape}")
print(f"Original test set:  {X_test.shape}")
print(f"Number of disease classes: {len(np.unique(y_train))}")

# ==================== Advanced Data Augmentation ====================
print("\n" + "="*60)
print("STARTING ADVANCED DATA AUGMENTATION")
print("="*60)

class_counts = Counter(y_train)
print(f"\nClass distribution BEFORE augmentation: {len(class_counts)} classes")

# Step 1: Aggressive Random Oversampling - balance to maximum class * 1.5
max_count = max(class_counts.values())
target_count = int(max_count * 1.5)
sampling_strategy = {cls: target_count for cls in class_counts.keys()}
ros = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
print(f"After ROS: {X_ros.shape[0]} samples")

# Step 2: SMOTE with k_neighbors=5
smote = SMOTE(random_state=42, k_neighbors=5, k_neighbors_default=5)
X_smote, y_smote = smote.fit_resample(X_ros, y_ros)
print(f"After SMOTE: {X_smote.shape[0]} samples")

# Step 3: Multi-intensity dropout augmentation
def multi_dropout_augmentation(X, y, drop_intensities=[0.05, 0.1, 0.15]):
    """Apply multiple levels of dropout augmentation."""
    X_augs = [X]
    y_augs = [y]
    
    for drop_prob in drop_intensities:
        X_aug = X.copy()
        mask = np.random.rand(*X_aug.shape) < drop_prob
        X_aug[mask] = 0
        X_augs.append(X_aug)
        y_augs.append(y.copy())
    
    return np.vstack(X_augs), np.hstack(y_augs)

X_dropout, y_dropout = multi_dropout_augmentation(X_smote, y_smote, drop_intensities=[0.05, 0.1, 0.15])
print(f"After dropout augmentation (3 levels): {X_dropout.shape[0]} samples")

# Step 4: Mixup augmentation
def mixup_augmentation(X, y, alpha=0.2, num_mixups=2000):
    """Create synthetic samples by mixing pairs of samples."""
    np.random.seed(42)
    n_samples = X.shape[0]
    X_mixup_list = []
    y_mixup_list = []
    
    for _ in range(num_mixups):
        idx1, idx2 = np.random.choice(n_samples, 2, replace=False)
        lam = np.random.beta(alpha, alpha)
        X_mixed = lam * X[idx1] + (1 - lam) * X[idx2]
        y_mixed = y[idx1] if np.random.rand() > 0.5 else y[idx2]
        X_mixup_list.append(X_mixed)
        y_mixup_list.append(y_mixed)
    
    return np.array(X_mixup_list), np.array(y_mixup_list)

X_mixup, y_mixup = mixup_augmentation(X_dropout, y_dropout, alpha=0.3, num_mixups=3000)
print(f"After mixup augmentation: {X_mixup.shape[0]} samples")

# Combine all augmented data
X_train_final = np.vstack([X_dropout, X_mixup])
y_train_final = np.hstack([y_dropout, y_mixup])
print(f"\nFINAL training set after ALL augmentation: {X_train_final.shape[0]} samples")

final_counts = Counter(y_train_final)
print(f"Final class distribution (per class): {min(final_counts.values())} - {max(final_counts.values())} samples")

# ==================== Train Individual Models ====================
print("\n" + "="*60)
print("TRAINING OPTIMIZED MODELS")
print("="*60)

trained_models = {}
results = []

# Model 1: XGBoost
print("\n[1/3] Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    min_child_weight=1,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)
xgb.fit(X_train_final, y_train_final)
y_pred_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')
trained_models['XGBoost'] = xgb
results.append({'Model': 'XGBoost', 'Accuracy': acc_xgb, 'F1 (weighted)': f1_xgb})
print(f"   XGBoost - Accuracy: {acc_xgb:.4f}, F1: {f1_xgb:.4f}")

# Model 2: Gradient Boosting
print("[2/3] Training Gradient Boosting...")
gb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    random_state=42
)
gb.fit(X_train_final, y_train_final)
y_pred_gb = gb.predict(X_test)
acc_gb = accuracy_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb, average='weighted')
trained_models['GradBoost'] = gb
results.append({'Model': 'Gradient Boosting', 'Accuracy': acc_gb, 'F1 (weighted)': f1_gb})
print(f"   GradBoost - Accuracy: {acc_gb:.4f}, F1: {f1_gb:.4f}")

# Model 3: Random Forest
print("[3/3] Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_final, y_train_final)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')
trained_models['RandomForest'] = rf
results.append({'Model': 'Random Forest', 'Accuracy': acc_rf, 'F1 (weighted)': f1_rf})
print(f"   RandomForest - Accuracy: {acc_rf:.4f}, F1: {f1_rf:.4f}")

# Model 4: Ensemble Voting
print("[4/4] Training Ensemble Voting...")
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', trained_models['XGBoost']),
        ('gb', trained_models['GradBoost']),
        ('rf', trained_models['RandomForest'])
    ],
    voting='soft'
)
y_pred_ensemble = voting_clf.predict(X_test)
acc_ensemble = accuracy_score(y_test, y_pred_ensemble)
f1_ensemble = f1_score(y_test, y_pred_ensemble, average='weighted')
trained_models['Ensemble'] = voting_clf
results.append({'Model': 'Ensemble Voting', 'Accuracy': acc_ensemble, 'F1 (weighted)': f1_ensemble})
print(f"   Ensemble - Accuracy: {acc_ensemble:.4f}, F1: {f1_ensemble:.4f}")

# ==================== Results Summary ====================
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

best_model = trained_models['Ensemble']
best_acc = acc_ensemble

print(f"\n✓ Best Model: Ensemble Voting Classifier")
print(f"✓ Test Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"✓ Test F1 Score: {f1_ensemble:.4f}")

# Save models
import os
os.makedirs('Models', exist_ok=True)
joblib.dump(best_model, 'Models/best_model.pkl')
joblib.dump(trained_models['XGBoost'], 'Models/xgboost_model.pkl')
joblib.dump(trained_models['GradBoost'], 'Models/gradboost_model.pkl')
joblib.dump(trained_models['RandomForest'], 'Models/rf_model.pkl')
print("\n✓ Models saved successfully!")

# Detailed classification report
print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT - ENSEMBLE MODEL")
print("="*60)
print(classification_report(y_test, y_pred_ensemble, target_names=le.classes_, zero_division=0))

ModuleNotFoundError: No module named 'xgboost'